In [ ]:
!pip install scikit-learn
!pip install pandas
!pip install sentencepiece

In [1]:
# filter dataset
!python3 MT-Preparation/filtering/filter.py ./en-zh.en ./en-zh.zh en zh

Dataframe shape (rows, columns): (231267, 2)
--- Rows with Empty Cells Deleted	--> Rows: 231267
--- Duplicates Deleted			--> Rows: 229646
--- Source-Copied Rows Deleted		--> Rows: 229640
--- Too Long Source/Target Deleted	--> Rows: 224743
--- HTML Removed			--> Rows: 224743
--- Rows will remain true-cased		--> Rows: 224743
--- Rows with Empty Cells Deleted	--> Rows: 224743
--- Rows Shuffled			--> Rows: 224743
--- Source Saved: ./en-zh.en-filtered.en
--- Target Saved: ./en-zh.zh-filtered.zh


In [5]:
from sklearn.feature_extraction.text import TfidfVectorizer
import numpy as np

lines = [
    "Thank you so much, Chris. And it's truly a great honor to have the opportunity to come to this stage twice; I'm extremely grateful.",
    "I have been blown away by this conference, and I want to thank all of you for the many nice comments about what I had to say the other night.",
    "And I say that sincerely, partly because  I need that.  Put yourselves in my position.",
    "I flew on Air Force Two for eight years.",
    "Now I have to take off my shoes or boots to get on an airplane!",
    "I'll tell you one quick story to illustrate what that's been like for me."
]

def extract_balanced_salient_words_tfidf(corpus, window_size=4, top_k=5):
    results = []
    n = len(corpus)

    for i in range(n):
        # Determine how many sentences to pull from before and after
        if i < window_size // 2:
            num_before = i
            num_after = window_size - num_before
        elif i > n - window_size // 2 - 1:
            num_after = n - i - 1
            num_before = window_size - num_after
        else:
            num_before = window_size // 2
            num_after = window_size // 2

        # Collect indices for context sentences (excluding the current one)
        before = list(range(max(0, i - num_before), i))
        after = list(range(i + 1, min(n, i + 1 + num_after)))
        context_indices = before + after
        pseudo_doc = [corpus[j] for j in context_indices]

        # If no context (corner case), append empty list
        if not pseudo_doc:
            results.append([])
            continue

        # Extract salient keywords with TF-IDF
        vectorizer = TfidfVectorizer(stop_words='english')
        tfidf = vectorizer.fit_transform(pseudo_doc)
        scores = np.asarray(tfidf.sum(axis=0)).flatten()
        feature_names = np.array(vectorizer.get_feature_names_out())

        top_indices = np.argsort(scores)[::-1][:top_k]
        salient_words = feature_names[top_indices].tolist()
        results.append(salient_words)

    return results

def write_salient_prefixed_raw_file(raw_lines, output_file, salient_word_lists):
    with open(output_file, "w") as f:
        for line, salient_words in zip(raw_lines, salient_word_lists):
            # Construct full line: salient words + separator + original sentence
            prefixed_line = ' '.join(salient_words + ['__SEP__'] + line.split())
            f.write(prefixed_line + '\n')

raw_lines = open("en-zh.en-filtered.en", "r").read().splitlines()
salient_contexts = extract_balanced_salient_words_tfidf(raw_lines, window_size=4, top_k=5)
write_salient_prefixed_raw_file(raw_lines, "en-zh.en-filtered-salient.en", salient_contexts)


[['say', 'airplane', 'boots', 'shoes', 'years'], ['shoes', 'airplane', 'boots', 'years', 'flew'], ['airplane', 'boots', 'shoes', 'thank', 'years'], ['say', 'airplane', 'shoes', 'boots', 'sincerely'], ['say', 'years', 'flew', 'force', 'air'], ['say', 'airplane', 'boots', 'shoes', 'years']]


In [ ]:
salient_words_per_sentence = extract_balanced_salient_words_tfidf(raw_lines)

In [2]:
# train a sentencepiece model for subwording
!python3 MT-Preparation/subwording/1-train_unigram.py ./en-zh.en-filtered.en ./en-zh.zh-filtered.zh

sentencepiece_trainer.cc(178) LOG(INFO) Running command: --input=./en-zh.en-filtered.en --model_prefix=source --vocab_size=10000 --hard_vocab_limit=false --split_digits=true --user_defined_symbols=__SEP__
sentencepiece_trainer.cc(78) LOG(INFO) Starts training with : 
trainer_spec {
  input: ./en-zh.en-filtered.en
  input_format: 
  model_prefix: source
  model_type: UNIGRAM
  vocab_size: 10000
  self_test_sample_size: 0
  character_coverage: 0.9995
  input_sentence_size: 0
  shuffle_input_sentence: 1
  seed_sentencepiece_size: 1000000
  shrinking_factor: 0.75
  max_sentence_length: 4192
  num_threads: 16
  num_sub_iterations: 2
  max_sentencepiece_length: 16
  split_by_unicode_script: 1
  split_by_number: 1
  split_by_whitespace: 1
  split_digits: 1
  pretokenization_delimiter: 
  treat_whitespace_as_suffix: 0
  allow_whitespace_only_pieces: 0
  user_defined_symbols: __SEP__
  required_chars: 
  byte_fallback: 0
  vocabulary_output_piece_score: 1
  train_extremely_large_corpus: 0
  see

In [3]:
# subword the dataset
!python3 MT-Preparation/subwording/2-subword.py source.model target.model ./en-zh.en-filtered.en ./en-zh.zh-filtered.zh

Source Model: source.model
Target Model: target.model
Source Dataset: ./en-zh.en-filtered.en
Target Dataset: ./en-zh.zh-filtered.zh
Done subwording the source file! Output: ./en-zh.en-filtered.en.subword
Done subwording the target file! Output: ./en-zh.zh-filtered.zh.subword


In [4]:
# first 3 lines before subwording
!head -n 3 ./en-zh.en-filtered.en && echo "-----" && head -n 3 ./en-zh.zh-filtered.zh

I found a couple in the work of Matt Groening, although Matt Groening told me later that he could not make my talk because it was a morning session and I gather that he is not an early riser.
And what we're doing, in essence, is we're teaching the dog, kind of like -- we're letting the dog think that the dog is training us.
And it's happening everywhere, among liberals and conservatives, agnostics and believers, the rich and the poor, East and West alike.
-----
我还在在迈特.格拉宁的作品中发现了一些， 尽管迈特.格拉宁之后告诉我 他不能来听我的演讲 因为是清晨的时间段 我知道他不是一个早起的人。
我们现在做的，本质上说，是在教狗，也有点像 —— 我们让狗认为，狗在训练我们。
而这中危险无处不在， 它存在于自由派和保守派中， 不可知论者和信徒之中，富人和穷人之中， 东方和西方之中。


In [5]:
# first 3 lines after subwording
!head -n 3 ./en-zh.en-filtered.en.subword && echo "---" && head -n 3 ./en-zh.zh-filtered.zh.subword after

▁I ▁found ▁a ▁couple ▁in ▁the ▁work ▁of ▁Matt ▁G ro ening , ▁although ▁Matt ▁G ro ening ▁told ▁me ▁later ▁that ▁he ▁could ▁not ▁make ▁my ▁talk ▁because ▁it ▁was ▁a ▁morning ▁session ▁and ▁I ▁gather ▁that ▁he ▁is ▁not ▁an ▁early ▁rise r .
▁And ▁what ▁we ' re ▁doing , ▁in ▁essence , ▁is ▁we ' re ▁teaching ▁the ▁dog , ▁kind ▁of ▁like ▁-- ▁we ' re ▁let ting ▁the ▁dog ▁think ▁that ▁the ▁dog ▁is ▁training ▁us .
▁And ▁it ' s ▁happening ▁everywhere , ▁among ▁liberal s ▁and ▁conservative s , ▁agnostic s ▁and ▁believe rs , ▁the ▁rich ▁and ▁the ▁poor , ▁East ▁and ▁West ▁a like .
---
==> ./en-zh.zh-filtered.zh.subword <==
▁我还 在 在 迈 特 . 格 拉 宁 的作品 中 发现了 一些 , ▁尽管 迈 特 . 格 拉 宁 之后 告诉我 ▁他 不能 来 听 我的演讲 ▁因为 是 清 晨 的时间 段 ▁我知道 他 不是一个 早 起 的人 。
▁我们现在 做的 , 本质上 说 , 是在 教 狗 , 也 有点像 ▁ —— ▁我们 让 狗 认为 , 狗 在 训练 我们 。
▁而这 中 危险 无处不在 , ▁它 存在于 自由 派 和 保守 派 中 , ▁ 不可 知 论 者 和 信 徒 之中 , 富 人 和 穷人 之中 , ▁ 东 方 和 西方 之中 。
head: cannot open 'after' for reading: No such file or directory


In [8]:
# split the dataset into training set, development set, and test set
# Development and test sets should be between 1000 and 5000 segments (here we chose 200)
!python3 MT-Preparation/train_dev_split/train_dev_test_split.py 2000 2000 ./en-zh.en-filtered.en.subword ./en-zh.zh-filtered.zh.subword

Dataframe shape: (224743, 2)
--- Empty Cells Deleted --> Rows: 224743
--- Wrote Files
Done!
Output files
./en-zh.en-filtered.en.subword.train
./en-zh.zh-filtered.zh.subword.train
./en-zh.en-filtered.en.subword.dev
./en-zh.zh-filtered.zh.subword.dev
./en-zh.en-filtered.en.subword.test
./en-zh.zh-filtered.zh.subword.test


In [6]:
!wc -l ./*.subword.*

     2000 ./en-zh.en-filtered-salient.en.subword.dev
     2000 ./en-zh.en-filtered-salient.en.subword.test
     2000 ./en-zh.en-filtered-salient.en.subword.test.desubword
   220743 ./en-zh.en-filtered-salient.en.subword.train
     2000 ./en-zh.en-filtered.en.subword.dev
     2000 ./en-zh.en-filtered.en.subword.test
     2000 ./en-zh.en-filtered.en.subword.test.desubword
   220743 ./en-zh.en-filtered.en.subword.train
      200 ./en-zh.en-filtered.en.subword.translated
      200 ./en-zh.en-filtered.en.subword.translated.desubword
     2000 ./en-zh.zh-filtered.zh.subword.dev
     2000 ./en-zh.zh-filtered.zh.subword.test
     2000 ./en-zh.zh-filtered.zh.subword.test.desubword
   220743 ./en-zh.zh-filtered.zh.subword.train
   680629 total


In [9]:
# check the first and last line from each dataset
!echo "---First line---"
!head -n 1 ./*.{train,dev,test}

!echo -e "\n---Last line---"
!tail -n 1 ./*.{train,dev,test}

---First line---
==> ./en-zh.en-filtered-salient.en.subword.train <==
▁came ▁mayor ▁dog ▁didn ▁west ▁ __SEP__ ▁I ▁found ▁a ▁couple ▁in ▁the ▁work ▁of ▁Matt ▁G ro en ing , ▁although ▁Matt ▁G ro en ing ▁told ▁me ▁later ▁that ▁he ▁could ▁not ▁make ▁my ▁talk ▁because ▁it ▁was ▁a ▁morning ▁session ▁and ▁I ▁gather ▁that ▁he ▁is ▁not ▁an ▁early ▁rise r .

==> ./en-zh.en-filtered.en.subword.train <==
▁I ▁found ▁a ▁couple ▁in ▁the ▁work ▁of ▁Matt ▁G ro ening , ▁although ▁Matt ▁G ro ening ▁told ▁me ▁later ▁that ▁he ▁could ▁not ▁make ▁my ▁talk ▁because ▁it ▁was ▁a ▁morning ▁session ▁and ▁I ▁gather ▁that ▁he ▁is ▁not ▁an ▁early ▁rise r .

==> ./en-zh.zh-filtered.zh.subword.train <==
▁我还 在 在 迈 特 . 格 拉 宁 的作品 中 发现了 一些 , ▁尽管 迈 特 . 格 拉 宁 之后 告诉我 ▁他 不能 来 听 我的演讲 ▁因为 是 清 晨 的时间 段 ▁我知道 他 不是一个 早 起 的人 。

==> ./en-zh.en-filtered-salient.en.subword.dev <==
▁mayor ▁friend ▁village ▁helped ▁shirt s ▁ __SEP__ ▁So ▁we ▁started ▁asking ▁ourselves : ▁What ▁kind ▁of ▁less ▁obvious ▁ metric s ▁could ▁we ▁use ▁to ▁actual